# Secondary-outcome calibration

Grouped calibration of SOC versus SOC+NMR at 1, 3, and 5 years for the T2D, CKD, and MASLD complication models. Predicted risks come from the saved fold-specific absolute-risk files.


In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
  library(survival)
  library(patchwork)
})


In [ ]:
# source oof calibration helper
source(here("helpers/oof_calibration.r"))


## Outcome configuration

Axis limits are the ones used for the secondary calibration figure. Each outcome points at the absolute-risk folder written by its model notebook.


In [ ]:
diabetes_risk_dir <- here("outputs/secondary_diabetes/absolute_risks")
kidney_risk_dir <- here("outputs/secondary_kidney/absolute_risks")
liver_risk_dir <- here("outputs/secondary_liver/absolute_risks")

table_dir <- here("outputs/tables")
figure_dir <- here("outputs/figures")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

horizons <- c(1, 3, 5)
n_groups <- 10
verbose_calibration <- FALSE

calib_palette <- c(
  "SOC"     = "#E64B35",
  "SOC+NMR" = "#4DBBD5"
)

family_order <- c("T2D", "CKD", "MASLD")

make_spec <- function(family, title, slug, risk_dir, axis_1yr, axis_3yr, axis_5yr) {
  list(
    family = family,
    title = title,
    slug = slug,
    risk_dir = risk_dir,
    soc_file = paste0(slug, "_marker_fold_baseline_risk.rds"),
    nmr_file = paste0(slug, "_marker_nmr_fold_baseline_risk.rds"),
    axis_limits = list(
      `1` = c(0, axis_1yr),
      `3` = c(0, axis_3yr),
      `5` = c(0, axis_5yr)
    )
  )
}

outcome_specs <- list(
  t2d_ckd = make_spec("T2D", "T2D to CKD", "t2d_ckd", diabetes_risk_dir, 0.25, 0.50, 0.75),
  t2d_nephropathy = make_spec("T2D", "T2D to Broad Renal Disease", "t2d_nephropathy", diabetes_risk_dir, 0.30, 0.60, 0.80),
  t2d_heart_failure = make_spec("T2D", "T2D to Heart Failure", "t2d_heart_failure", diabetes_risk_dir, 0.10, 0.25, 0.40),
  t2d_neuropathy = make_spec("T2D", "T2D to Neuropathy", "t2d_neuropathy", diabetes_risk_dir, 0.15, 0.30, 0.40),
  t2d_foot = make_spec("T2D", "T2D to Foot Pathology", "t2d_foot", diabetes_risk_dir, 0.10, 0.15, 0.25),
  t2d_retinopathy = make_spec("T2D", "T2D to Retinopathy", "t2d_retinopathy", diabetes_risk_dir, 0.15, 0.25, 0.40),

  ckd_heart_failure = make_spec("CKD", "CKD to Heart Failure", "ckd_heart_failure", kidney_risk_dir, 0.15, 0.30, 0.45),
  ckd_chd = make_spec("CKD", "CKD to Coronary Heart Disease", "ckd_chd", kidney_risk_dir, 0.10, 0.20, 0.35),
  ckd_afib = make_spec("CKD", "CKD to Atrial Fibrillation", "ckd_afib", kidney_risk_dir, 0.10, 0.30, 0.40),
  ckd_esrd = make_spec("CKD", "CKD to Kidney Failure", "ckd_esrd", kidney_risk_dir, 0.20, 0.40, 0.60),
  ckd_acs = make_spec("CKD", "CKD to Acute MI/ACS", "ckd_acs", kidney_risk_dir, 0.10, 0.35, 0.55),
  ckd_tia_stroke = make_spec("CKD", "CKD to TIA/Stroke", "ckd_tia_stroke", kidney_risk_dir, 0.10, 0.25, 0.40),

  nafld_diab = make_spec("MASLD", "MASLD to Type 2 Diabetes", "nafld_diab", liver_risk_dir, 0.30, 0.45, 0.60),
  nafld_ckd = make_spec("MASLD", "MASLD to CKD", "nafld_ckd", liver_risk_dir, 0.20, 0.45, 0.70),
  nafld_portalht = make_spec("MASLD", "MASLD to Portal Hypertension", "nafld_portalht", liver_risk_dir, 0.05, 0.10, 0.15),
  nafld_venous_thrombosis = make_spec("MASLD", "MASLD to Venous Thrombosis", "nafld_venous_thrombosis", liver_risk_dir, 0.05, 0.10, 0.15)
)


## Load and plot

Each spec is read from its own absolute-risk folder. SOC and SOC+NMR must cover the same people.


In [ ]:
oof_risk_column <- function(t_star) {
  paste0("pred_risk_", as.integer(t_star), "yr")
}

load_disease_oof_risks <- function(spec) {
  soc_path <- file.path(spec$risk_dir, spec$soc_file)
  nmr_path <- file.path(spec$risk_dir, spec$nmr_file)
  missing <- c(soc_path, nmr_path)[!file.exists(c(soc_path, nmr_path))]
  if (length(missing) > 0) {
    stop("Missing absolute-risk RDS file(s):\n", paste(missing, collapse = "\n"))
  }

  objects <- list(soc = readRDS(soc_path), nmr = readRDS(nmr_path))
  soc_id <- objects$soc$oof_predictions$StudyID
  nmr_id <- objects$nmr$oof_predictions$StudyID
  if (!setequal(soc_id, nmr_id)) {
    stop(spec$title, " SOC and SOC+NMR cohorts differ.")
  }
  objects
}

plot_disease_oof_calibration <- function(
  spec,
  oof_objects,
  t_star,
  n_groups = 10,
  palette,
  verbose = FALSE
) {
  risk_col <- oof_risk_column(t_star)
  axis_limits <- spec$axis_limits[[as.character(t_star)]]
  if (is.null(axis_limits)) {
    stop("No axis_limits for '", spec$title, "' at t_star = ", t_star, ".")
  }

  cal_soc <- compute_calibration_df_from_oof_risk(
    oof_objects$soc,
    t_star = t_star,
    risk_col = risk_col,
    n_groups = n_groups,
    verbose = verbose
  )
  cal_nmr <- compute_calibration_df_from_oof_risk(
    oof_objects$nmr,
    t_star = t_star,
    risk_col = risk_col,
    n_groups = n_groups,
    verbose = verbose
  )

  plot <- plot_calibration_compare(
    list("SOC" = cal_soc, "SOC+NMR" = cal_nmr),
    title = spec$title,
    t_star = t_star,
    axis_limits = axis_limits,
    show_error_bars = TRUE,
    show_lines = TRUE,
    palette = palette
  )

  metrics <- bind_rows(
    tibble(model = "SOC", calibration = list(cal_soc)),
    tibble(model = "SOC+NMR", calibration = list(cal_nmr))
  ) %>%
    mutate(
      family = spec$family,
      outcome = spec$title,
      slug = spec$slug,
      horizon_years = t_star,
      n = map_int(calibration, ~ attr(.x, "n")),
      n_events = map_int(calibration, ~ attr(.x, "n_events")),
      mean_abs_calib_error = map_dbl(calibration, ~ attr(.x, "mean_abs_calib_error")),
      grouped_ici = map_dbl(calibration, ~ attr(.x, "ici")),
      calibration_score = map_dbl(calibration, ~ attr(.x, "calibration_score")),
      max_abs_calib_error = map_dbl(calibration, ~ attr(.x, "max_abs_calib_error"))
    ) %>%
    select(
      family, outcome, slug, horizon_years, model, n, n_events,
      mean_abs_calib_error, grouped_ici, calibration_score, max_abs_calib_error
    )

  groups <- bind_rows("SOC" = cal_soc, "SOC+NMR" = cal_nmr, .id = "model") %>%
    mutate(
      family = spec$family,
      outcome = spec$title,
      slug = spec$slug,
      horizon_years = t_star,
      .before = 1
    )

  list(plot = plot, metrics = metrics, groups = groups)
}


## Preflight and load


In [ ]:
missing_files <- unlist(lapply(outcome_specs, function(spec) {
  paths <- file.path(spec$risk_dir, c(spec$soc_file, spec$nmr_file))
  paths[!file.exists(paths)]
}), use.names = FALSE)

if (length(missing_files) > 0) {
  stop("Missing absolute-risk RDS file(s):\n", paste(missing_files, collapse = "\n"))
}

oof_store <- lapply(outcome_specs, load_disease_oof_risks)
names(oof_store) <- names(outcome_specs)
cat("Loaded", length(oof_store), "secondary outcomes.\n")


## Calibration results

Metrics and group-level values are written under `outputs/tables`.


In [ ]:
calibration_results <- setNames(lapply(horizons, function(t_star) {
  setNames(lapply(names(outcome_specs), function(id) {
    plot_disease_oof_calibration(
      spec = outcome_specs[[id]],
      oof_objects = oof_store[[id]],
      t_star = t_star,
      n_groups = n_groups,
      palette = calib_palette,
      verbose = verbose_calibration
    )
  }), names(outcome_specs))
}), as.character(horizons))

calibration_metrics <- bind_rows(lapply(calibration_results, function(by_outcome) {
  bind_rows(lapply(by_outcome, `[[`, "metrics"))
})) %>%
  mutate(
    family = factor(family, levels = family_order),
    outcome = factor(outcome, levels = map_chr(outcome_specs, "title")),
    model = factor(model, levels = c("SOC", "SOC+NMR"))
  ) %>%
  arrange(family, outcome, horizon_years, model)

calibration_groups <- bind_rows(lapply(calibration_results, function(by_outcome) {
  bind_rows(lapply(by_outcome, `[[`, "groups"))
})) %>%
  mutate(
    family = factor(family, levels = family_order),
    outcome = factor(outcome, levels = map_chr(outcome_specs, "title")),
    model = factor(model, levels = c("SOC", "SOC+NMR"))
  ) %>%
  arrange(family, outcome, horizon_years, model, group)

write.csv(
  calibration_metrics,
  file.path(table_dir, "secondary_calibration_metrics.csv"),
  row.names = FALSE
)
write.csv(
  calibration_groups,
  file.path(table_dir, "secondary_calibration_groups.csv"),
  row.names = FALSE
)

calibration_metrics


## Family figures

One row per family at each horizon. The 5-year figures are shown below. Files go to `outputs/figures`.


In [ ]:
assemble_family_grid <- function(results_at_horizon, family, t_star) {
  ids <- names(outcome_specs)[map_chr(outcome_specs, "family") == family]
  plots <- lapply(ids, function(id) {
    results_at_horizon[[id]]$plot +
      theme(plot.title = element_text(size = 11, face = "plain", hjust = 0.5))
  })

  wrap_plots(plots, nrow = 1, guides = "collect") +
    plot_annotation(
      title = paste0(family, " secondary-outcome calibration (", t_star, " year)"),
      theme = theme(plot.title = element_text(size = 16, face = "bold", hjust = 0.5))
    ) &
    theme(legend.position = "bottom")
}

calibration_figures <- list()
for (t_star in horizons) {
  for (family in family_order) {
    key <- paste(tolower(family), t_star, "yr", sep = "_")
    p <- assemble_family_grid(calibration_results[[as.character(t_star)]], family, t_star)
    calibration_figures[[key]] <- p

    dims <- if (family == "MASLD") c(width = 14, height = 3.6) else c(width = 20, height = 3.6)
    ggsave(
      file.path(figure_dir, paste0("secondary_calibration_", tolower(family), "_", t_star, "yr.png")),
      plot = p,
      width = dims[["width"]],
      height = dims[["height"]],
      dpi = 600
    )
  }
}


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 3.6)
calibration_figures$t2d_5_yr


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 3.6)
calibration_figures$ckd_5_yr


In [ ]:
options(repr.plot.width = 14, repr.plot.height = 3.6)
calibration_figures$masld_5_yr
